# Canary-1B-v2 & Parakeet-TDT-0.6B-v3 — Demo de Inferencia
**Procesamiento de Datos Secuenciales con Deep Learning** | Londoño · Rojas · García · Ayala | NVIDIA (Sekoyan et al., 2025)

> Activar GPU: Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU

## 1. Entorno

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode == 0 else 'No GPU — activar T4 en Entorno de ejecucion')
!python --version

## 2. Instalacion de dependencias

~3-5 minutos la primera vez.

In [ ]:
# NeMo requiere Python >=3.10 <3.12; Colab usa 3.10 — compatible
print('Instalando NeMo toolkit...')
!pip install nemo_toolkit[asr] soundfile librosa -q 2>&1 | tail -3
print('Instalacion completada.')

## 3. Imports

In [ ]:
import os, time, tempfile, io
import numpy as np
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt
import IPython.display as ipd
import nemo.collections.asr as nemo_asr
print('Imports OK')

## 4. Carga de modelos

In [ ]:
# Parakeet-TDT-0.6B-v3 (~1.2 GB) — ASR unicamente
print('Cargando Parakeet-TDT-0.6B-v3...')
t0 = time.time()
parakeet = nemo_asr.models.EncDecRNNTBPEModel.from_pretrained('nvidia/parakeet-tdt-0.6b-v3')
parakeet.eval()
print(f'Parakeet listo en {time.time()-t0:.1f}s | '
      f'{sum(p.numel() for p in parakeet.parameters())/1e6:.0f}M params')

In [ ]:
# Canary-1B-v2 (~2 GB) — ASR + AST en 25 idiomas
print('Cargando Canary-1B-v2...')
t0 = time.time()
canary = nemo_asr.models.EncDecMultiTaskModel.from_pretrained('nvidia/canary-1b-v2')
canary.eval()
# Decodificacion greedy para mayor velocidad
decode_cfg = canary.cfg.decoding
decode_cfg.beam.beam_size = 1
canary.change_decoding_strategy(decode_cfg)
print(f'Canary listo en {time.time()-t0:.1f}s | '
      f'{sum(p.numel() for p in canary.parameters())/1e6:.0f}M params')

## 5. Carga de audio

In [ ]:
# Opcion A: audio de muestra (LibriSpeech, ingles)
!wget -q -O sample.wav https://www.signalogic.com/melp/EngSamples/Orig/male.wav
info = sf.info('sample.wav')
print(f'Muestra: {info.duration:.1f}s | {info.samplerate}Hz | {info.channels}ch')
ipd.display(ipd.Audio('sample.wav'))

In [ ]:
# Opcion B: subir tu propio archivo de audio
from google.colab import files
uploaded = files.upload()
if uploaded:
    fname = list(uploaded.keys())[0]
    data, sr = sf.read(fname)
    if data.ndim > 1:
        data = data.mean(axis=1)   # convertir a mono
    sf.write('custom.wav', data, sr)
    print(f'Audio cargado: {fname} -> custom.wav ({len(data)/sr:.1f}s)')
    ipd.display(ipd.Audio('custom.wav'))

In [ ]:
# Seleccionar archivo a usar en el resto del notebook
AUDIO_PATH = 'custom.wav' if os.path.exists('custom.wav') else 'sample.wav'
AUDIO_LANG = 'en'  # cambiar si el audio esta en otro idioma
duration = sf.info(AUDIO_PATH).duration
print(f'Audio activo: {AUDIO_PATH} ({duration:.1f}s)')

## 6. Inferencia

In [ ]:
# Parakeet — transcripcion ASR
t0 = time.time()
res_pk = parakeet.transcribe([AUDIO_PATH])
t_pk = time.time() - t0
txt_pk = res_pk[0] if isinstance(res_pk[0], str) else res_pk[0].text
print(f'[Parakeet ASR]')
print(f'  Texto  : {txt_pk}')
print(f'  Tiempo : {t_pk:.2f}s | RTFx: {duration/t_pk:.1f}x')

In [ ]:
# Canary — transcripcion ASR
t0 = time.time()
res_cn = canary.transcribe(
    audio=[AUDIO_PATH], batch_size=1,
    task='asr', source_lang=AUDIO_LANG, target_lang=AUDIO_LANG, pnc='no'
)
t_cn = time.time() - t0
txt_cn = res_cn[0] if isinstance(res_cn[0], str) else res_cn[0].text
print(f'[Canary ASR]')
print(f'  Texto  : {txt_cn}')
print(f'  Tiempo : {t_cn:.2f}s | RTFx: {duration/t_cn:.1f}x')

In [ ]:
# Canary — traduccion AST (en -> es)
if AUDIO_LANG == 'en':
    t0 = time.time()
    res_ast = canary.transcribe(
        audio=[AUDIO_PATH], batch_size=1,
        task='ast', source_lang='en', target_lang='es', pnc='no'
    )
    t_ast = time.time() - t0
    txt_ast = res_ast[0] if isinstance(res_ast[0], str) else res_ast[0].text
    print(f'[Canary AST en->es]')
    print(f'  Texto  : {txt_ast}')
    print(f'  Tiempo : {t_ast:.2f}s | RTFx: {duration/t_ast:.1f}x')
else:
    print('Cambiar AUDIO_LANG a "en" para probar la traduccion')

## 7. Visualizacion de resultados

In [ ]:
# Tabla comparativa
print(f'{"Modelo":<30} {"Tiempo":>8} {"RTFx":>8}  Resultado')
print('-'*80)
print(f'{"Parakeet-TDT-0.6B-v3":<30} {t_pk:>7.2f}s {duration/t_pk:>7.1f}x  {txt_pk[:40]}')
print(f'{"Canary-1B-v2 (ASR)":<30} {t_cn:>7.2f}s {duration/t_cn:>7.1f}x  {txt_cn[:40]}')
if AUDIO_LANG == 'en':
    print(f'{"Canary-1B-v2 (AST en->es)":<30} {t_ast:>7.2f}s {duration/t_ast:>7.1f}x  {txt_ast[:40]}')

In [ ]:
# Grafica RTFx — modelos del paper vs. experimento actual
nombres   = ['Whisper\nv3', 'Voxtral\nMini', 'Phi-4\nMM', 'Parakeet\nv3', 'Canary\nv2']
wer_ref   = [7.44, 7.05, 6.14, 6.32, 7.15]
rtfx_ref  = [145, 110, 62, 3333, 749]
colores   = ['#6c8a9e','#6c8a9e','#6c8a9e','#06D6A0','#0077B6']

fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor='#0a1628')
for ax in axes:
    ax.set_facecolor('#0d2137')
    for sp in ax.spines.values(): sp.set_edgecolor('#1c3a52')

# WER
bars = axes[0].barh(nombres, wer_ref, color=colores, height=0.55)
axes[0].set_xlabel('WER (%) — menor es mejor', color='#90e0ef', fontsize=9)
axes[0].set_title('Word Error Rate', color='#00b4d8', fontsize=11)
axes[0].tick_params(colors='#90e0ef'); axes[0].set_xlim(0, 9)
for b, v in zip(bars, wer_ref):
    axes[0].text(v+0.1, b.get_y()+b.get_height()/2, f'{v}%', va='center', color='white', fontsize=8)

# RTFx
bars2 = axes[1].barh(nombres, rtfx_ref, color=colores, height=0.55)
axes[1].set_xlabel('RTFx — mayor es mejor', color='#90e0ef', fontsize=9)
axes[1].set_title('Velocidad de Inferencia', color='#00b4d8', fontsize=11)
axes[1].tick_params(colors='#90e0ef')
for b, v in zip(bars2, rtfx_ref):
    axes[1].text(v+20, b.get_y()+b.get_height()/2, f'{v:,}x', va='center', color='white', fontsize=8)
# Linea del experimento actual
axes[1].axvline(duration/t_pk, color='#f4a261', lw=1.5, ls='--')
axes[1].text(duration/t_pk+30, -0.5, f'Tu sesion\n{duration/t_pk:.0f}x', color='#f4a261', fontsize=7)

plt.tight_layout()
plt.savefig('rtfx_benchmark.png', dpi=130, bbox_inches='tight', facecolor='#0a1628')
plt.show()

In [ ]:
# Mel-spectrogram — representacion de entrada al FastConformer encoder
y, sr = librosa.load(AUDIO_PATH, sr=16000, mono=True)
mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=80, fmax=8000)
mel_db = librosa.power_to_db(mel, ref=np.max)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), facecolor='#0a1628')
for ax in axes:
    ax.set_facecolor('#0d2137')
    for sp in ax.spines.values(): sp.set_edgecolor('#1c3a52')

# Forma de onda
t = np.linspace(0, len(y)/sr, len(y))
axes[0].plot(t, y, color='#00b4d8', lw=0.5, alpha=0.8)
axes[0].set_title('Forma de onda (16kHz)', color='#00b4d8', fontsize=10)
axes[0].set_xlabel('Tiempo (s)', color='#90e0ef'); axes[0].tick_params(colors='#90e0ef')

# Mel-spectrogram
img = librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel',
                                fmax=8000, ax=axes[1], cmap='magma')
axes[1].set_title(
    f'Mel-Spectrogram [80 filtros] — entrada al FastConformer '
    f'({mel_db.shape[1]} frames -> {mel_db.shape[1]//8} despues del subsampling 8x)',
    color='#00b4d8', fontsize=9)
axes[1].set_xlabel('Tiempo (s)', color='#90e0ef'); axes[1].tick_params(colors='#90e0ef')
fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
plt.tight_layout()
plt.savefig('melspectrogram.png', dpi=130, bbox_inches='tight', facecolor='#0a1628')
plt.show()
print(f'Dimensiones mel: {mel_db.shape} -> {mel_db.shape[1]//8} frames tras subsampling 8x')

## 8. Interfaz grafica Streamlit

Requiere token gratuito de ngrok: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# Instalar Streamlit y pyngrok
!pip install streamlit pyngrok -q
print('Instalacion completada.')

In [ ]:
APP_CODE = """
import os, io, time, tempfile
import streamlit as st
import numpy as np
import soundfile as sf

st.set_page_config(page_title="ASR/AST Demo", layout="wide")

@st.cache_resource(show_spinner=False)
def load_parakeet():
    import nemo.collections.asr as nemo_asr
    m = nemo_asr.models.EncDecRNNTBPEModel.from_pretrained("nvidia/parakeet-tdt-0.6b-v3")
    m.eval()
    return m

@st.cache_resource(show_spinner=False)
def load_canary():
    import nemo.collections.asr as nemo_asr
    m = nemo_asr.models.EncDecMultiTaskModel.from_pretrained("nvidia/canary-1b-v2")
    m.eval()
    cfg = m.cfg.decoding
    cfg.beam.beam_size = 1
    m.change_decoding_strategy(cfg)
    return m

def save_audio(f):
    data, sr = sf.read(io.BytesIO(f.read()))
    if data.ndim > 1:
        data = data.mean(axis=1)
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
    sf.write(tmp.name, data, sr)
    return tmp.name, len(data) / sr

LANGS = {"en":"Ingles","es":"Espanol","fr":"Frances","de":"Aleman",
         "it":"Italiano","pt":"Portugues","ru":"Ruso","nl":"Holandes",
         "pl":"Polaco","uk":"Ucraniano","cs":"Checo","da":"Danes",
         "fi":"Finlandes","el":"Griego","hr":"Croata","hu":"Hungaro",
         "lt":"Lituano","lv":"Leton","ro":"Rumano","sk":"Eslovaco",
         "sl":"Esloveno","sv":"Sueco","bg":"Bulgaro","et":"Estonio","mt":"Maltes"}

st.title("Canary-1B-v2 & Parakeet-TDT-0.6B-v3 — Demo de Inferencia")
st.caption("FastConformer + Transformer Decoder | NVIDIA (Sekoyan et al., 2025)")
st.caption("Procesamiento de Datos Secuenciales con Deep Learning")

with st.sidebar:
    st.header("Configuracion")
    modelo = st.radio("Modelo", ["Parakeet-TDT-0.6B-v3", "Canary-1B-v2", "Ambos"])
    tarea  = st.radio("Tarea (Canary)", ["ASR - Transcripcion", "AST - Traduccion"])
    lang_keys = list(LANGS.keys())
    src = st.selectbox("Idioma del audio", lang_keys, format_func=lambda k: LANGS[k])
    tgt = src
    if "AST" in tarea:
        tgt = st.selectbox("Idioma destino", lang_keys, format_func=lambda k: LANGS[k], index=1)
    st.markdown("---")
    st.markdown("**Canary-1B-v2**  RTFx=749 | WER=7.15%")
    st.markdown("**Parakeet**       RTFx=3332 | WER=6.32%")

tab1, tab2, tab3 = st.tabs(["Inferencia", "Arquitectura", "Atencion Q/K/V"])

with tab1:
    f = st.file_uploader("Archivo de audio", type=["wav","mp3","flac","ogg"])
    if f:
        st.audio(f)
        path, dur = save_audio(f)
        st.caption(f"Duracion: {dur:.1f}s")
        if st.button("Ejecutar inferencia", type="primary"):
            if modelo in ["Parakeet-TDT-0.6B-v3","Ambos"]:
                with st.spinner("Cargando Parakeet..."):
                    pk = load_parakeet()
                t0 = time.time()
                r = pk.transcribe([path])
                t = time.time()-t0
                txt = r[0] if isinstance(r[0],str) else r[0].text
                st.success(f"**Parakeet ASR:** {txt}")
                c1,c2 = st.columns(2)
                c1.metric("Tiempo", f"{t:.2f}s"); c2.metric("RTFx", f"{dur/t:.1f}x")
            if modelo in ["Canary-1B-v2","Ambos"]:
                with st.spinner("Cargando Canary..."):
                    cn = load_canary()
                task_key = "ast" if "AST" in tarea else "asr"
                t0 = time.time()
                r = cn.transcribe([path], batch_size=1, task=task_key,
                                  source_lang=src, target_lang=tgt, pnc="no")
                t = time.time()-t0
                txt = r[0] if isinstance(r[0],str) else r[0].text
                label = f"AST {src}->{tgt}" if task_key=="ast" else "ASR"
                st.success(f"**Canary {label}:** {txt}")
                c1,c2 = st.columns(2)
                c1.metric("Tiempo", f"{t:.2f}s"); c2.metric("RTFx", f"{dur/t:.1f}x")
        os.unlink(path)

with tab2:
    st.code('''
Audio WAV 16kHz
    |
Mel-Spectrogram [T x 80]
    |
FastConformer Encoder
    |-- Conv Subsampling 8x     [T/8 x d_model]
    |-- Linear Projection
    |-- x24 Conformer Blocks
    |     LayerNorm > FF(0.5) > MHSA > Conv > FF(0.5) > LayerNorm
    |
Transformer Decoder (autoregresivo)
    |-- Token Embedding
    |-- Self-Attention (causal)
    |-- Cross-Attention  Q=decoder / K,V=encoder
    |-- FFN > Softmax
    |
Texto generado (token por token)
''', language='')
    c1,c2,c3,c4 = st.columns(4)
    c1.metric("Params Canary","1B"); c2.metric("Params Parakeet","600M")
    c3.metric("Idiomas","25"); c4.metric("Entrenamiento","1.7M h audio")

with tab3:
    st.latex(r"\\text{Attention}(Q,K,V)=\\text{softmax}\\!\\left(\\frac{QK^T}{\\sqrt{d_k}}\\right)V")
    st.code('''
def attention(Q, K, V, mask=None):
    # Q, K, V: [batch, heads, seq, d_k]
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2,-1) / d_k**0.5  # similitud
    if mask is not None:
        scores = scores.masked_fill(mask==0, float('-inf'))
    weights = scores.softmax(dim=-1)             # normalizacion
    return weights @ V                           # extraccion
''', language='python')
    st.markdown('''
| Matriz | Origen | Funcion |
|--------|--------|---------|
| Q (Query) | Decoder / Encoder | Define que informacion se busca |
| K (Key)   | Encoder           | Describe el contenido disponible |
| V (Value) | Encoder           | Informacion extraida |

**ALiBi Simetrico (innovacion del articulo):**
`score(i,j) = Q·K/sqrt(d_k) - m*|i-j|`
Primera aplicacion de ALiBi bidireccional en ASR/AST.
''')
"""
with open("app.py","w") as f:
    f.write(APP_CODE)
print(f"app.py escrito ({len(APP_CODE)} chars)")

In [ ]:
import subprocess, time, urllib.request, sys

# Instalar pyngrok si no esta disponible
try:
    from pyngrok import ngrok, conf
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyngrok', '-q'], check=True)
    from pyngrok import ngrok, conf

# Pegar token de https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "TU_TOKEN_AQUI"
conf.get_default().auth_token = NGROK_TOKEN

# Matar procesos previos
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
subprocess.run(['pkill', '-f', 'ngrok'], capture_output=True)
time.sleep(2)

# Arrancar Streamlit en background
subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Esperar hasta que Streamlit responda
print('Esperando Streamlit...')
for i in range(20):
    try:
        urllib.request.urlopen('http://localhost:8501', timeout=1)
        print(f'Streamlit listo ({i+1}s)')
        break
    except:
        time.sleep(1)

# Abrir tunel ngrok
tunnel = ngrok.connect(8501)
print(f'\nURL: {tunnel.public_url}')

In [ ]:
# Detener Streamlit y cerrar tunel
import subprocess
from pyngrok import ngrok
ngrok.disconnect(tunnel.public_url)
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
print('Detenido.')